Static: bar graph of the distribution of the shapes of the UFOs

In [34]:
# imports
import altair as alt
import pandas as pd

# disable max rows limit for altair so more of data can be used
alt.data_transformers.disable_max_rows()

# read data
data = pd.read_csv("archive/scrubbed.csv", low_memory=False)

# transform datetime column to datetime (currently stored as a string)
data["datetime"] = pd.to_datetime(data["datetime"], errors="coerce")

# filter to only US data
project_df = data[data["country"] == "us"].copy()

# filter data from 2005 onwards (window of 10 years of data)
recent_df = project_df[project_df["datetime"].dt.year >= 2005].copy()

recent_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
143,2005-10-10 07:40:00,seattle,wa,us,other,60,one + minutes,round symetrical with roundish flat bottom shi...,10/11/2005,47.6063889,-122.330833
144,2005-10-10 09:00:00,apache junction,az,us,other,30,30 seconds,half-moon shaped objects that just winked out,10/11/2005,33.4150000,-111.548889
145,2005-10-10 09:30:00,north miami beach,fl,us,oval,1800,30 minutes,i sent you an e-mail 2 days ago and really wan...,10/20/2005,25.9327778,-80.162778
146,2005-10-10 14:45:00,los angeles,ca,us,egg,10,10 seconds,Egg UFO over Hollywood Hills and LAX in LOS AN...,10/20/2005,34.0522222,-118.242778
147,2005-10-10 20:00:00,loretto,pa,us,triangle,300,5 minutes,Dull red flash&#44 Triangular ship&#44 Vocal N...,10/30/2006,40.5030556,-78.630556


In [35]:
# Extra cleaning and aggregate top 10 shapes 

# Clean the shape column 
recent_df["shape"] = recent_df["shape"].astype(str).str.strip().str.lower()

# Remove missing/blank/unknown-like values
shape_df = recent_df[
    recent_df["shape"].notna() &
    (recent_df["shape"] != "") &
    (recent_df["shape"] != "nan") &
    (recent_df["shape"] != "unknown") &
    (recent_df["shape"] != "other")
].copy()

# Aggregate counts
shape_counts = (
    shape_df["shape"]
    .value_counts()
    .head(10)
    .reset_index()
)

# Rename columns
shape_counts.columns = ["shape", "count"]

shape_counts

,shape,count
0,light,8788
1,circle,4116
2,triangle,3753
3,fireball,3645
4,sphere,2777
5,oval,1816
6,disk,1689
7,formation,1266
8,changing,992
9,cigar,839


In [36]:
# Interactive Altair bar graph of distribution of shapes of UFOs

# Create a hover selection
hover = alt.selection_point(
    fields=["shape"],
    on="mouseover",
    empty="none"
)

# Bar chart
bar_chart = (
    alt.Chart(shape_counts)
    .mark_bar(cornerRadiusTopLeft=6, cornerRadiusTopRight=6)
    .encode(
        x=alt.X(
            "count:Q",
            title="Number of Sightings"
        ),
        y=alt.Y(
            "shape:N",
            sort="-x",
            title="Shape"
        ),
        color=alt.condition(
            hover,
            alt.value("#39ff14"),   
            alt.value("#2d1b4e")   
        ),
        tooltip=[
            alt.Tooltip("shape:N", title="Shape"),
            alt.Tooltip("count:Q", title="Sightings", format=",")
        ]
    )
    .properties(
        title="Top 10 UFO Shapes in U.S. Sightings (2005 Onward)",
        width=700,
        height=400
    )
    .add_params(hover)
)

# Text labels at end of bars
text = (
    alt.Chart(shape_counts)
    .mark_text(
        align="left",
        baseline="middle",
        dx=5,
        fontSize=12, 
        color="white"
    )
    .encode(
        x=alt.X("count:Q"),
        y=alt.Y("shape:N", sort="-x"),
        text=alt.Text("count:Q", format=",")
    )
)

final_chart = (bar_chart + text).configure_axis(
    labelFontSize=12,
    titleFontSize=13,
    labelColor="black",
    titleColor="black"
).configure_title(
    fontSize=16,
    anchor="start",
    color="black"
).configure_view(
    fill="#0b0b1a"   # dark space background
)

final_chart

alt.LayerChart(...)

In [37]:
# Save chart
final_chart.save("Top_10_UFO_Shapes.html")

## Heatmap for UFO Sighting Occurrences by Days of the Week

In [38]:
# make a copy of dataframe
heatmap_df = recent_df.copy()

# Remove rows where datetime is missing
heatmap_df = heatmap_df.dropna(subset=["datetime"]).copy()

# Extract day of week and hour
heatmap_df["day_of_week"] = heatmap_df["datetime"].dt.day_name()
heatmap_df["time"] = heatmap_df["datetime"].dt.strftime("%-I:00 %p")

# Set weekday order
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

# Aggregate counts by day and hour
weekday_hour_counts = (
    heatmap_df.groupby(["day_of_week", "time"])
    .size()
    .reset_index(name="count")
)

weekday_hour_counts.head()

,day_of_week,time,count
0,Friday,10:00 AM,94
1,Friday,10:00 PM,829
2,Friday,11:00 AM,87
3,Friday,11:00 PM,597
4,Friday,12:00 AM,285


In [ ]:
# Create interactive heatmap
hover_heat = alt.selection_point(
    fields=["day_of_week", "hour"],
    on="mouseover",
    empty="none"
)

heatmap = (
    alt.Chart(weekday_hour_counts)
    .mark_rect(stroke="#1a1a2e", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "day_of_week:N",
            sort=day_order,
            title="Day of Week"
        ),
        y=alt.Y(
            "time:O",
            sort=[
                "12:00 AM","1:00 AM","2:00 AM","3:00 AM","4:00 AM","5:00 AM",
                "6:00 AM","7:00 AM","8:00 AM","9:00 AM","10:00 AM","11:00 AM",
                "12:00 PM","1:00 PM","2:00 PM","3:00 PM","4:00 PM","5:00 PM",
                "6:00 PM","7:00 PM","8:00 PM","9:00 PM","10:00 PM","11:00 PM"
            ],
            title="Time of Day"
        ),
        color=alt.Color(
            "count:Q",
            scale=alt.Scale(
                range=["#0b0b3b", "#3a0ca3", "#b5179e", "#4cc9f0"]
            ),
            legend=alt.Legend(title="Number of Sightings")
        ),
        opacity=alt.condition(hover_heat, alt.value(1), alt.value(0.85)),
        tooltip=[
            alt.Tooltip("day_of_week:N", title="Day"),
            alt.Tooltip("time:O", title="Time of Day"),
            alt.Tooltip("count:Q", title="Sightings", format=",")
        ]
    )
    .properties(
        title="UFO Sightings by Day of Week and Time of Day",
        width=500,
        height=350
    )
    .add_params(hover_heat)
    .configure_axis(
        labelFontSize=12,
        titleFontSize=13,
        labelColor="black",
        titleColor="black"
    )
    .configure_title(
        fontSize=16,
        anchor="start",
        color="black"
    )
    .configure_view(
        fill="#0b0b1a"
    )
)

heatmap

alt.Chart(...)